# ExoBaby — Long-Tailed Distribution Validation (Qwen3.5-9B)

**Model:** `Qwen/Qwen3.5-9B` — 4-bit NF4, float16 (same config as `qwen3_5_9b_word_miner`)
**Inference style:** Closed-list detection (ExoBaby_LongTailed_Colab style)
**Vocabulary:** User-configurable CDI path (set `VOCAB_CSV` in Step 1)

## Workflow
1. Mount Drive and set paths (including CDI vocab path)
2. Collect all frames — **excluding** the 1,000 frames used to mine the CDI vocabulary
3. Load Qwen3.5-9B with 4-bit NF4 (exact word-miner config)
4. Run closed-list detection: show CDI vocab → model says which items are visible
5. Fit power-law, generate figures, OOI coverage, provenance (identical to `ExoBaby_LongTailed_Colab`)

## Key design decisions
| Aspect | Detail |
|---|---|
| Model | `Qwen/Qwen3.5-9B` (same as word miner) |
| Quantization | 4-bit NF4, float16, double quant (same as word miner) |
| `enable_thinking` | `False` — suppresses reasoning chain for clean output |
| Frame exclusion | Mining frames removed (prevents circular evaluation) |
| CDI vocab | User-specified path — choose 377 or 551 category file |
| Prompt | Closed-list: show CDI vocab, ask which items are visible |
| Output CSV | Identical schema to YOLOE+CLIP — all downstream steps unchanged |

---

## Step 0 — Clone repo & install packages

In [ ]:
REPO_URL  = 'https://github.com/tanzim12911/ExoBaby-pipeline.git'
REPO_NAME = 'ExoBaby-pipeline'

import os, subprocess
if not os.path.isdir(f'/content/{REPO_NAME}'):
    subprocess.run(['git', 'clone', REPO_URL, f'/content/{REPO_NAME}'], check=True)
    print('Repo cloned.')
else:
    subprocess.run(['git', '-C', f'/content/{REPO_NAME}', 'pull'], check=True)
    print('Repo already present, pulled latest.')
os.chdir(f'/content/{REPO_NAME}')
print('Working directory:', os.getcwd())

In [ ]:
# Install in the same order as qwen3_5_9b_word_miner
!pip uninstall -y torchaudio
!pip install -q --upgrade pip
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q \
    accelerate \
    bitsandbytes \
    qwen-vl-utils \
    sentencepiece \
    torchvision \
    pillow \
    pandas \
    tqdm
# Analytics dependencies
!pip install -q powerlaw scipy matplotlib numpy huggingface_hub
print('All packages installed.')

> ⚠️ **RESTART SESSION after the cell above completes.**
> Click: **Runtime → Restart session**, then continue from Step 1.
> (Required because transformers was installed from source.)

---
## Step 1 — Imports, Drive mount & path configuration

**Edit the variables in the configuration block below.**

| Variable | What to set |
|---|---|
| `FRAMES_DIR` | Path to your `filtered/` frames folder in Drive |
| `OUTPUT_BASE` | Root folder for all ExoBaby outputs |
| `RUN_TAG` | Label for this run (affects output subfolder name) |
| `VOCAB_CSV` | CDI vocabulary file — choose 377 or 551 categories |
| `MINING_LOG` | Path to the word-miner frame log (frames to exclude) |
| `PRUNE_STALE` | Set `True` to remove CSV rows for deleted frames |

In [ ]:
import os
import re
import csv
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.notebook import tqdm
from PIL import Image

warnings.filterwarnings('ignore')
plt.rcParams.update({'font.family': 'sans-serif', 'font.size': 12})

from transformers import AutoProcessor, BitsAndBytesConfig
try:
    from transformers import AutoModelForImageTextToText as QwenModelClass
except ImportError:
    from transformers import AutoModelForVision2Seq as QwenModelClass
from qwen_vl_utils import process_vision_info

print('Imports OK.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── CONFIGURATION ─────────────────────────────────────────
FRAMES_DIR  = '/content/drive/MyDrive/data/filtered'
OUTPUT_BASE = '/content/drive/MyDrive/ExoBaby-results'
RUN_TAG     = 'qwen35_9b'   # change per CDI run, e.g. 'qwen35_9b_377' or 'qwen35_9b_551'

# CDI vocabulary — pick one:
VOCAB_CSV = 'CDI/Bangla/stage4_concrete_objects_qwen3_5_9B_CDI_377.csv'
# VOCAB_CSV = 'CDI/Bangla/stage4_concrete_objects_qwen3_5_9B_CDI_551.csv'

# Frame exclusion log (frames used to mine the CDI vocabulary)
MINING_LOG  = 'CDI/Bangla/Qwen3_5_9B_mining_sampled_frames_log.txt'

PRUNE_STALE = False   # True = remove CSV rows for frames no longer on disk
# ──────────────────────────────────────────────────

import detection.config as cfg
from run_detection import resolve_shared_dirs, resolve_analysis_dirs

shared_dirs   = resolve_shared_dirs(OUTPUT_BASE)
analysis_dirs = resolve_analysis_dirs(OUTPUT_BASE, RUN_TAG)

dirs = {
    'data':       shared_dirs['data'],
    'frame_data': shared_dirs['frame_data'],
    'results':    analysis_dirs['results'],
    'figures':    analysis_dirs['figures'],
}

# Detection CSV name encodes the vocab used (keeps runs separate on Drive)
vocab_tag     = os.path.splitext(os.path.basename(VOCAB_CSV))[0]
detection_csv = os.path.join(shared_dirs['frame_data'], f'detections_{vocab_tag}.csv')

print(f'Frames dir      : {FRAMES_DIR}')
print(f'Vocab CSV       : {VOCAB_CSV}')
print(f'Detection CSV   : {detection_csv}')
print(f'Analysis output : {OUTPUT_BASE}/analysis/{RUN_TAG}/')
print(f'Prune stale     : {PRUNE_STALE}')

if not os.path.isdir(FRAMES_DIR):
    print(f'\n  WARNING: FRAMES_DIR not found: {FRAMES_DIR}')
    print('  Edit FRAMES_DIR above to match your Drive path.')
else:
    print('\n  Frames folder found.')
    if os.path.exists(detection_csv):
        n_existing = len(pd.read_csv(detection_csv))
        print(f'  Existing detection CSV: {n_existing:,} rows (will resume from here)')

---
## Step 2 — Collect frames & exclude mining set

All frames in `FRAMES_DIR` are collected, then the **1,000 frames used to mine the CDI**
vocabulary are excluded. This prevents circular evaluation — the model is tested on
frames it has never described before.

The exclusion log is matched on the relative path `video_id/chunk/frame.jpg`,
so it works regardless of where `FRAMES_DIR` is mounted on Colab.

In [ ]:
from run_detection import collect_frames
from detection.qwen35_detector import load_mining_exclusions

# All frames on disk
all_frame_paths = collect_frames(FRAMES_DIR)
print(f'Total frames found: {len(all_frame_paths):,}')

# Load the mining exclusion set
mining_exclusions = load_mining_exclusions(MINING_LOG)
print(f'Mining exclusion frames: {len(mining_exclusions):,}')

# Filter: keep only frames NOT in the mining set
def _rel(path, base):
    return os.path.relpath(path, base).replace('\\', '/')

frame_paths = [
    p for p in all_frame_paths
    if _rel(p, FRAMES_DIR) not in mining_exclusions
]

excluded = len(all_frame_paths) - len(frame_paths)
print(f'Excluded (mining): {excluded:,}')
print(f'Frames for inference: {len(frame_paths):,}')
if frame_paths:
    print(f'  First: {frame_paths[0]}')
    print(f'  Last : {frame_paths[-1]}')

---
## Step 3 — Load CDI vocabulary

Categories are sorted by **`presence_count` descending** before being passed to the model.
This ensures the most frequently annotated objects appear first in the prompt,
mitigating the "lost in the middle" attention bias on long lists.

In [ ]:
if not os.path.exists(VOCAB_CSV):
    raise FileNotFoundError(
        f'CDI file not found: {VOCAB_CSV}\n'
        'Check that VOCAB_CSV points to the correct file.'
    )

df_vocab = pd.read_csv(VOCAB_CSV)

# Sort descending by presence_count — most frequent categories first
df_vocab = df_vocab.sort_values('presence_count', ascending=False).reset_index(drop=True)

included_categories = df_vocab['english_label'].str.strip().str.lower().tolist()
lemma_to_semantic   = (
    df_vocab.drop_duplicates(subset=['english_label'])
            .set_index('english_label')['category']
            .str.strip().str.lower()
            .to_dict()
)

print(f'Vocabulary : {VOCAB_CSV}')
print(f'Categories : {len(included_categories)}')
print(f'Domains    : {sorted(df_vocab["category"].unique())}')
print(f'Top 10 (most frequent) : {included_categories[:10]}')
print(f'Bottom 10 (rarest)     : {included_categories[-10:]}')

---
## Step 4 — GPU check & runtime estimate

Run this before loading the model to confirm T4 is available.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')

if device == 'cuda':
    props   = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {vram_gb:.1f} GB')
else:
    print()
    print('  No GPU detected.')
    print('  Go to: Runtime > Change runtime type > T4 GPU > Save, then re-run.')

# Runtime estimate: ~4-6 sec/frame on T4 (9B model, 4-bit, 512x512 input)
n_frames      = len(frame_paths)
secs_per_frame = 5.0
est_hours     = n_frames * secs_per_frame / 3600

print()
print('── Runtime estimate ──────────────────────────────────')
print(f'  Frames for inference : {n_frames:,}')
print(f'  Est. time (T4)       : {est_hours:.1f} hours ({secs_per_frame:.0f} sec/frame)')
print(f'  Colab sessions       : ~{max(1, int(est_hours / 11))} session(s) of ~12 h each')
print('  Tip: detection CSV is resumable — re-run Step 6 after reconnect.')
print('──────────────────────────────────────────────────')

---
## Step 5 — Load Qwen3.5-9B (4-bit NF4)

**Configuration is identical to `qwen3_5_9b_word_miner.ipynb`:**
- `AutoModelForImageTextToText` (VLM class)
- 4-bit NF4, `float16`, `double_quant=True`
- `enable_thinking=False` — suppresses chain-of-thought reasoning for clean CSV output

**VRAM on T4 (16 GB):**
- Weights (4-bit NF4, 9B params): ~5.5 GB
- KV cache + activations: ~4–5 GB
- **Total: ~9–11 GB** — fits T4

In [ ]:
from detection.qwen35_detector import load_qwen35_model

model, processor = load_qwen35_model(device=device)
print('Qwen3.5-9B (4-bit NF4) loaded and ready.')

if device == 'cuda':
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    print(f'VRAM: {allocated:.2f} GB allocated / {reserved:.2f} GB reserved')

---
## Step 6 — Run detection

> **Resumable:** Re-run this cell after any disconnect. Already-processed frames are skipped.

**Per frame:**
1. The CDI vocabulary (sorted most-frequent-first) is shown to the model
2. The model replies with a comma-separated list of visible categories
3. The response is matched against the CDI set (exact lowercase match)
4. Matched categories are written to the detection CSV

In [ ]:
from detection.qwen35_detector import run_qwen35_detection

run_qwen35_detection(
    frame_paths=frame_paths,
    included_categories=included_categories,   # sorted by frequency DESC
    output_csv=detection_csv,
    model=model,
    processor=processor,
    device=device,
    prune_stale=PRUNE_STALE,
)

print(f'\nDetection CSV: {detection_csv}')

In [ ]:
df_raw = pd.read_csv(detection_csv)
df_det_all = df_raw[df_raw['class_name'].notna() & (df_raw['class_name'] != '')].copy()

print(f'Total CSV rows         : {len(df_raw):,}')
print(f'Sentinel rows (no det) : {len(df_raw) - len(df_det_all):,}')
print(f'Total detections       : {len(df_det_all):,}')
print(f'Unique frames hit      : {df_det_all["frame_path"].nunique():,} / {len(frame_paths):,}')
print(f'Unique categories      : {df_det_all["class_name"].nunique()} / {len(included_categories)}')
print('\nTop 20 detected categories:')
print(df_det_all['class_name'].value_counts().head(20).to_string())

---
## Step 7 — Long-tailed distribution analysis

Identical to `ExoBaby_LongTailed_Colab` from this step onwards.

In [ ]:
from detection.analysis import build_category_summary, fit_power_law, print_summary

intermediate_csv = os.path.join(
    dirs['results'],
    f'long_tailed_dist_{vocab_tag}.csv',
)

df_cat = build_category_summary(
    detection_csv=detection_csv,
    included_categories=included_categories,
    lemma_to_semantic=lemma_to_semantic,
    total_frames=len(frame_paths),
    output_csv=intermediate_csv,
)

print(f'Categories with >=1 detection: {(df_cat["count_frames"] > 0).sum()} / {len(included_categories)}')
print('\nTop 10:')
print(df_cat[['category', 'count_detected', 'count_frames', 'proportion', 'cdi_semantic']].head(10).to_string(index=False))

In [ ]:
fit = fit_power_law(df_cat)

print(f'Power-law exponent alpha : {fit["alpha"]:.3f}')
print(f'xmin                     : {fit["xmin"]:.1f}')
print(f'R (PL vs lognormal)      : {fit["r_vs_lognormal"]:.3f}  p = {fit["p_vs_lognormal"]:.4f}')
verdict = 'Power-law preferred' if fit['r_vs_lognormal'] > 0 else 'Lognormal preferred'
print(f'Verdict                  : {verdict}')

---
## Step 8 — Figures

In [ ]:
import detection.visualizer as vis

# Override plot_per_domain to cap each domain at top-15 bars.
# Critical for large CDI sets (household has 100+ categories in the 551 file).
def plot_per_domain_clean(df_cat, figures_dir: str, dataset_label: str = 'ExoBaby') -> str:
    present   = [d for d in vis.CDI_SEMANTIC_ORDER if d in df_cat['cdi_semantic'].values]
    ncols     = 4
    nrows     = -(-len(present) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3.5 * nrows))
    axes_flat = axes.flatten() if nrows > 1 else list(axes)
    for ax, domain in zip(axes_flat, present):
        sub   = df_cat[(df_cat['cdi_semantic'] == domain) & (df_cat['proportion'] > 0)].sort_values('proportion', ascending=False)
        color = vis.CDI_SEMANTIC_COLORS.get(domain, '#8B9A9E')
        if len(sub) == 0:
            ax.text(0.5, 0.5, 'No Detections', ha='center', va='center', color='gray', fontsize=11)
            ax.set_xticks([])
        else:
            sub = sub.head(15)
            ax.bar(range(len(sub)), sub['proportion'], color=color, width=0.85, linewidth=0)
            ax.set_xticks(range(len(sub)))
            ax.set_xticklabels(sub['category'].tolist(), rotation=45, ha='right', fontsize=9)
        ax.set_title(domain.replace('_', ' ').title(), fontsize=11, color=color, fontweight='bold')
        vis._style_ax(ax)
    for ax in axes_flat[len(present):]:
        ax.set_visible(False)
    fig.suptitle(f'Per-Domain Object Distribution ({dataset_label})', fontsize=14, y=1.01)
    fig.tight_layout()
    out_path = vis._save_figure(fig, figures_dir, f'fig2_per_domain_{vocab_tag}.png')
    return out_path

vis.plot_per_domain = plot_per_domain_clean
print('plot_per_domain overridden (top-15 per domain cap).')

In [ ]:
from detection.visualizer import plot_rank_bar, plot_loglog, plot_per_domain

total_detected = len(df_det_all)
dataset_label  = f'ExoBaby — Qwen3.5-9B ({vocab_tag})'

bar_path = plot_rank_bar(df_cat, fit, len(frame_paths), total_detected, dirs['figures'],
                         dataset_label=dataset_label)
print(f'Saved: {bar_path}')

In [ ]:
log_path = plot_loglog(fit['nonzero_df'], fit['counts_arr'], fit['alpha'], dirs['figures'])
print(f'Saved: {log_path}')

In [ ]:
dom_path = plot_per_domain(df_cat, dirs['figures'], dataset_label=dataset_label)
print(f'Saved: {dom_path}')

---
## Step 9 — Results summary

In [ ]:
print_summary(df_cat, fit, len(frame_paths), total_detected)

---
## Step 10 — Frame-level OOI coverage

Proportion of held-out frames that contain at least one CDI Object of Interest.

In [ ]:
frames_with_detections = df_det_all['frame_path'].nunique()
total_frames           = len(frame_paths)
proportion             = frames_with_detections / total_frames if total_frames else 0.0

print(f'Total held-out frames  : {total_frames:,}')
print(f'Frames with detections : {frames_with_detections:,}')
print(f'OOI coverage           : {proportion:.2%}')

ooi_stats = {
    'run_tag':                RUN_TAG,
    'model':                  'Qwen3.5-9B (4-bit NF4)',
    'vocabulary':             os.path.basename(VOCAB_CSV),
    'total_held_out_frames':  total_frames,
    'mining_excluded_frames': len(all_frame_paths) - total_frames,
    'frames_with_detections': frames_with_detections,
    'proportion':             round(proportion, 6),
    'proportion_pct':         f'{proportion:.2%}',
}

ooi_json = os.path.join(dirs['results'], f'ooi_coverage_{vocab_tag}.json')
with open(ooi_json, 'w', encoding='utf-8') as f:
    json.dump(ooi_stats, f, indent=2)
print(f'\nSaved: {ooi_json}')

---
## Step 11 — Provenance analysis

For each CDI domain, identify the top-5 categories by detection count
and break down each by source video.

> If one video contributes >70% of detections for a category, that category's
> frequency reflects that video's setting — not a general pattern. This is a dataset
> bias check.

> This step only reads from the saved CSV and does not require the GPU.
> You can re-run it independently at any time.

In [ ]:
from detection.analysis import build_provenance

provenance_csv = os.path.join(
    dirs['results'],
    f'provenance_top{cfg.TOP_N_PER_DOMAIN}_per_domain_{vocab_tag}.csv',
)

# Rebuild df_cat if this cell is run in a new session (no GPU needed)
if 'df_cat' not in dir():
    from run_detection import collect_frames
    from detection.qwen35_detector import load_mining_exclusions
    _all   = collect_frames(FRAMES_DIR)
    _excl  = load_mining_exclusions(MINING_LOG)
    _fps   = [p for p in _all if os.path.relpath(p, FRAMES_DIR).replace('\\', '/') not in _excl]
    df_cat = build_category_summary(
        detection_csv=detection_csv,
        included_categories=included_categories,
        lemma_to_semantic=lemma_to_semantic,
        total_frames=len(_fps),
        output_csv=os.path.join(dirs['results'], f'long_tailed_dist_{vocab_tag}.csv'),
    )

df_prov = build_provenance(
    detection_csv=detection_csv,
    df_cat=df_cat,
    output_csv=provenance_csv,
    top_n=cfg.TOP_N_PER_DOMAIN,
)

print(f'Provenance CSV : {provenance_csv}')
print(f'  Rows         : {len(df_prov):,}')
print(f'  Unique videos: {df_prov["video_id"].nunique()}')
print()
print(df_prov.head(20).to_string(index=False))

In [ ]:
from detection.visualizer import plot_provenance

prov_path = plot_provenance(df_prov, cfg.TOP_N_PER_DOMAIN, dirs['figures'])
print(f'Saved: {prov_path}')

---
## Output structure

```
ExoBaby-results/
├── frame_data/
│   └── detections_stage4_concrete_objects_qwen3_5_9B_CDI_377.csv
└── analysis/
    └── qwen35_9b/              ← set by RUN_TAG
        ├── results/
        │   ├── long_tailed_dist_<vocab_tag>.csv
        │   ├── provenance_top5_per_domain_<vocab_tag>.csv
        │   └── ooi_coverage_<vocab_tag>.json
        └── figures/
            ├── fig1a_long_tailed_bar_valid129.png  (.pdf)
            ├── fig1b_loglog_valid129.png  (.pdf)
            └── fig2_per_domain_<vocab_tag>.png
```

To run with the **551-category** vocab: change `VOCAB_CSV` and `RUN_TAG` in Step 1 and re-run from Step 3.